# Download CPG0016 Metadata and Referenced Files
This notebook shows how to discover and download all `load_data_with_illum.csv` files for `cpg0016`, concatenate them into one pandas DataFrame, filter that DataFrame, and then download referenced `.npy` illumination files or `.tif` image files. Download calls now require both an explicit metadata DataFrame and a column containing per-row output directories. The downloader automatically excludes `source_all`.

In [ ]:
from pathlib import Path

from jump_image_datasets.cpg0016 import CPG0016LoadDataWithIllumDownloader

## Download all `load_data_with_illum.csv` files
Each CSV is stored under its source-relative S3 path inside the local directory you provide so files from different runs or plates do not collide.

In [ ]:
# Download all load_data_with_illum.csv files for cpg0016, excluding source_all.
# This can take a while even though the total size is only ~15 GB because it pulls
# ~2.5k separate S3 CSV objects; per-file request latency, public-s3fs overhead,
# Python copy/write overhead, and later CSV parsing dominate more than raw bandwidth.
# The total time for this process was 2 hours, 30 minutes with 8 cores in parallel with the S3 check.
# The time to download the csvs only took 10 to 15 minutes
# If you already downloaded the csvs, then you can disable the check.
downloader = CPG0016LoadDataWithIllumDownloader(
    csv_download_dir=Path("downloaded_cpg0016_csvs"),
    parallel=True,
    workers=8,
    use_existing_csvs_without_s3_check=True,
)
print(downloader.csv_download_summary)

## Concatenate the downloaded CSVs
`get_dataframe()` reads all downloaded `load_data_with_illum.csv` files and appends provenance columns so you can trace rows back to the source CSV.

In [ ]:
metadata_df = downloader.get_dataframe()
print(f"Metadata shape: {metadata_df.shape}")
metadata_df[["Metadata_Source", "Metadata_Batch", "Metadata_Plate"]].head()

## Filter metadata and assign output folders
Download methods require an explicit DataFrame plus an output directory column. Relative and absolute output paths are both supported.

In [ ]:
filtered_metadata_df = metadata_df.iloc[:10].copy()
filtered_metadata_df["OutputDir"] = (
    filtered_metadata_df["Metadata_Plate"].astype(str).radd("downloaded_cpg0016_images/")
)
print(f"Filtered metadata shape: {filtered_metadata_df.shape}")

orig_dna_summary = downloader.download_files_from_column(
    dataframe=filtered_metadata_df,
    column_name="URL_OrigDNA",
    output_dir_column="OutputDir",
    parallel=True,
    workers=8,
)
print(orig_dna_summary)

## Download grouped illumination `.npy` files
Use `download_illumination_files(...)` to download the standard illumination columns in one call from your filtered metadata DataFrame.

In [ ]:
illum_summary = downloader.download_illumination_files(
    dataframe=filtered_metadata_df,
    output_dir_column="OutputDir",
    parallel=True,
    workers=8,
)
print(illum_summary)

## Download grouped original image `.tif` files
Use `download_image_files(...)` to download the standard original image columns in one call from your filtered metadata DataFrame.

In [ ]:
image_summary = downloader.download_image_files(
    dataframe=filtered_metadata_df,
    output_dir_column="OutputDir",
    parallel=True,
    workers=8,
)
print(image_summary)